<a href="https://colab.research.google.com/github/oliwialosko/ML_Assignment1/blob/main/assignments/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oliwialosko/ML_Assignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [ ]:
fact_daily = con.sql(f"SELECT * FROM {TABLES['fact_daily']} LIMIT 10").df()
fact_daily.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [ ]:
fact_daily.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [ ]:
fact_query = con.sql(f"SELECT * FROM {TABLES['fact_query_90d']} LIMIT 10").df()
fact_query.head()

,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


In [ ]:
fact_query.columns

Index(['client_hash_id', 'content_hash_id', 'query_hash_id',
       'query_char_count', 'query_token_count', 'window_start', 'window_end',
       'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30',
       'impressions_prev30', 'clicks_prev30', 'avg_position_90d',
       'avg_position_last30', 'avg_position_prev30',
       'content_total_impressions_90d', 'content_visible_query_count',
       'rare_query_count', 'rare_impressions_share',
       'anonymized_impressions_share'],
      dtype='object')

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row equals exactly one content item on the page per client per day. We want to aggregate it to make it one row per one item per client.

For defining the features I choose time window restricted to March 2026 (2026-03) and I will predict on the second half of the month

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

For modeling I have to create new features based on already existing ones in the fact_daily table. These are the features already sorted into four buckets.

Features: imp_past, clicks_past, avg_pos_past, sessions_past, visible_days_past. Sums or averages of impressions, clicks etc. Knowable at the decision moment. Agregated from the first half of the month.

Label / Proxy: is_declining_label (1 if future impressions dropped significantly). Computed from the second half of the month. Never used as a feature.

Context: client_hash_id, content_hash_id. Used only to identify the item of analysis.

Excluded: imp_future: because it contains future information (what we are trying to predict) and would cause data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass
import duckdb
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

In [ ]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

TARGET_MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print("Date Span Verification")
span_query = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           MIN(report_date) as min_date,
           MAX(report_date) as max_date
    FROM read_parquet('{TARGET_MONTH}')
""").df()
print(span_query)

print("\n Availability Verification")
avail_query = con.sql(f"""
    SELECT COUNT(*) as rows_with_ga4
    FROM read_parquet('{TARGET_MONTH}')
    WHERE ga4_data_available IS TRUE
""").df()
print(avail_query)

Date Span Verification


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

 Availability Verification


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_with_ga4
0         413966


In [ ]:
first_grain_check = con.sql(f"""
    SELECT *
    FROM read_parquet('{TARGET_MONTH}')
    LIMIT 1
""").df()
print(first_grain_check)

  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True           False                True                <NA>   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               20           0                67  ...         <NA>   

   ai_chatgpt  ai_perplexity  ai_gemini  ai_copilot  ai_claude  ai_meta  \
0        <NA>           <NA>       <NA>        <NA>       <NA>     <NA>   

   ai_other  scroll_events    month  
0      <NA>           <NA>  2026-03  

[1 rows x 31 columns]


In [ ]:
# 5 feature frame and grain verifiaction
# divide march in two halfs - first and the second (until 15th)
# because we are focused on impressons we only create imp_future for prediction
feature_query = f"""
    SELECT client_hash_id,
           content_hash_id,
           SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_past,
           SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_past,
           SUM(CASE WHEN report_date <= '2026-03-15' THEN ga4_sessions ELSE 0 END) AS sessions_past,
           AVG(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position END) AS avg_pos_past,
           COUNT(DISTINCT CASE WHEN report_date <= '2026-03-15' THEN report_date END) AS visible_days_past,

           SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_future
    FROM read_parquet('{TARGET_MONTH}')
    GROUP BY client_hash_id, content_hash_id
    HAVING imp_past > 50  -- only the pages with any movement
"""
features_df = con.sql(feature_query).df()

grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, COUNT(*) c
    FROM features_df
    GROUP BY client_hash_id, content_hash_id
    HAVING c > 1 LIMIT 5
""").df()
print(f"Duplicate grain rows found: {len(grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


In [ ]:
print(grain_check)

Empty DataFrame
Columns: [client_hash_id, content_hash_id, c]
Index: []


In [ ]:
print(f"Total distinct content items: {len(features_df)}")
display(features_df.head())

Total distinct content items: 92133


,client_hash_id,content_hash_id,imp_past,clicks_past,sessions_past,avg_pos_past,visible_days_past,imp_future
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,0.0,0.0,5.222776,15,70.0
1,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,1.0,0.0,3.737399,15,680.0
2,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,0.0,0.0,6.156643,15,1614.0
3,client_62f4a7e64f5e0096,content_d49a012dcb924e31,246.0,0.0,0.0,4.520919,15,83.0
4,client_62f4a7e64f5e0096,content_614baf2af4330bd7,413.0,1.0,0.0,4.390322,15,359.0


In [ ]:
print("\n Experiment with the leakage")
# label/proxy : is_declining (if the impressions have declined under 80% of what was in the past)
features_df["is_declining_label"] = (features_df["imp_future"] < 0.8 * features_df["imp_past"]).astype(int)

leakage_features = ["imp_past", "clicks_past", "sessions_past", "avg_pos_past", "visible_days_past", "imp_future"]
X_leaky = features_df[leakage_features].fillna(0)
y = features_df["is_declining_label"]

leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
preds_leaky = leaky_model.predict(X_leaky)
print(f"Leaky Model Precision: {precision_score(y, preds_leaky, zero_division=0):.3f}")

#honest model - NO LEAKAGE
honest_features = ["imp_past", "clicks_past", "sessions_past", "avg_pos_past", "visible_days_past"]
X_honest = features_df[honest_features].fillna(0)

honest_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
preds_honest = honest_model.predict(X_honest)
print(f"Honest Model Precision: {precision_score(y, preds_honest, zero_division=0):.3f}")


 Experiment with the leakage
Leaky Model Precision: 0.904
Honest Model Precision: 0.558


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced History: Clients might have connected their GSC and GA4 accounts at different dates and therefore have zeros before which might be misleading for the model.

GSC vs. GA4 Mismatch: Some clients have GSC-only early rows because they connected GSC before GA4. During those mismatch periods, GA4 metrics are not zero but unmeasured and unknown. Therefore, the data cannot reliably tell us the true conversion rate from impressions to sessions for those windows.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.